# The parallelization floor, measured on your GPU

Companion to [jmurray10/agent_design_peas](https://github.com/jmurray10/agent_design_peas), which argues that classical
agent architectures survive the arrival of LLMs, and that an LLM replaces one
component inside them rather than the architecture itself.

One directory in that repository is about parallelism rather than agents. Its
claim: **every parallel execution strategy has a size below which it is slower
than doing the work in order**, because setup is paid per launch while the saving
is proportional to the work.

The repository measures that on a CPU with the standard library. It cannot measure
it on a GPU, because the machine it was written on does not have one. This notebook
is that missing half, and the numbers it prints are **yours**, from whatever GPU
Colab allocated you.

No API key. No model is called anywhere in this notebook. It is arithmetic, timed.

---

**Before you run it:** Runtime -> Change runtime type -> Hardware accelerator ->
GPU. The first cell checks, and says so if you forgot.

## 1. What hardware did you get


In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                          '--format=csv'], capture_output=True, text=True).stdout)
else:
    print('No GPU attached to this runtime.')
    print('Runtime -> Change runtime type -> Hardware accelerator -> GPU, then')
    print('run this cell again. The CPU half of the argument further down')
    print('works either way.')


## 2. Get the code

The measurement is one file in the repository, `06-parallelization/gpu_floor.py`.
The Hugging Face Space imports the same module, so a number quoted from either
front end came out of the same implementation.

In [ ]:
!git clone --depth 1 https://github.com/jmurray10/agent_design_peas.git peas 2>/dev/null || echo 'already cloned'
%cd peas

## 3. The CPU half, which needs nothing at all

This is the claim as the repository makes it with no GPU and no installs: a pure
Python kernel run sequentially against a process pool, across a range of sizes.
Below the crossover, parallel loses outright.

In [ ]:
!python 06-parallelization/benchmark_floor.py

## 4. The same shape on your GPU

Three columns per row, and the middle one is the honest one:

- **cpu ms** -- numpy on the Colab CPU
- **gpu ms** -- the GPU counting both transfers, which is what offloading costs
- **kernel ms** -- the GPU counting only the kernel

The kernel-only column is the one that gets quoted. The column beside it is what
you actually get if your data starts on the host and the answer has to come back.

In [ ]:
import sys
sys.path.insert(0, '06-parallelization')
from gpu_floor import measure

print(measure())

## What to take from it

On the run behind the repository's own write-up, on a Blackwell slice, the
element-wise operation **never crossed over**: counting both transfers the CPU
finished first at every size from a thousand elements to thirty million, on the same
rows where the kernel-only column read 32x to 76x. The matrix multiply did cross
over, at n = 1024.

Your numbers will differ, and the interesting question is whether that *structure*
holds on your hardware: element-wise late or never, matrix multiply early. The
reason it should is that arithmetic in a matrix multiply grows as n cubed while the
transfers grow as n squared, so the ratio moves in the GPU's favour as the problem
grows. Element-wise work has no such asymmetry to exploit.

A kernel-only speedup is a true statement about the kernel. It is not a number you
can spend, unless your data already lives on the device and stays there.

**Provenance.** Nothing here reproduces a published figure. The source material
behind the repository quotes 33x for SAXPY and 437x for a tiled matrix multiply,
cited there as published figures and measured by neither the repository nor this
notebook. What you just ran is one sample, on one allocation, on one day. numpy
reaches several cores through BLAS, so the CPU column is not a single-threaded
baseline and these ratios are not core-count ratios.